<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/llm_zero_shot_classifier_with_llama_ccp_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Illustration of an llm-based zero-shot classifier using the llama-cpp-python classifier

In [1]:
# install the llama-ccp-python library from source with CUDA support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 284.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 252.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 313.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 280.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 259.7 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.36-py3-none-linux_x86_64.whl size=146235709 sha256=8f66ccf63c00f25c503cf072e4904ece17aab4c7897e1a78036c304f3b80e93b
  Stored in directory: /tmp/pip-ephem-wheel-cache-s8ok2vly/wheels/e5/bf/ad/f5f50f854633df40f54d89e4e5271e987476036203819f8b4c
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation:

In [2]:
from enum import Enum
import json
from typing import Dict, Tuple
from llama_cpp import Llama
from pydantic import BaseModel, Field, create_model

In [3]:
# 2. Load a quantized Gemma GGUF model directly from Hugging Face
# n_gpu_layers=-1 offloads the entire model to the Colab GPU
llm = Llama.from_pretrained(
    repo_id="bartowski/gemma-4-12B-it-GGUF",
    filename="gemma-4-12B-it-Q8_0.gguf",
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gemma-4-12B-it-Q8_0.gguf: reconstructing file:   0%|          |  0.00B / 12.7GB            

./gemma-4-12B-it-Q8_0.gguf: downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 2048


In [4]:
def classify_text(
    llm: Llama,
    categories: Dict[str, str],
    text_fragment: str
) -> Tuple[str, float]:
    """
    Classifies a text fragment into one of the provided categories with guaranteed schema enforcement.

    Parameters:
        llm (Llama): The initialized Llama instance.
        categories (Dict[str, str]): A dict of {category_name: description}.
        text_fragment (str): The piece of text to classify.

    Returns:
        Tuple[str, float]: A tuple containing (chosen_category_name, confidence_score).
    """
    # 1. Dynamically create an Enum from the dictionary keys to enforce valid choices
    CategoryEnum = Enum("CategoryEnum", {k: k for k in categories.keys()})

    # 2. Build the dynamic Pydantic schema
    class ClassificationResponse(BaseModel):
        category: CategoryEnum = Field(description="The category that best fits the text fragment.")
        confidence: float = Field(description="Confidence score between 0.0 and 1.0.")

    # 3. Format the categories into a readable string for the prompt
    categories_formatted = "\n".join(
        [f"- {name}: {desc}" for name, desc in categories.items()]
    )

    # 4. Construct the prompt using Gemma's formatting
    prompt = f"""<start_of_turn>user
You are a precise text classification assistant. Analyze the text fragment provided below and assign it to the single best category from the allowed list. Use the category descriptions to guide your choice.

Allowed Categories:
{categories_formatted}

Text Fragment to classify:
"{text_fragment}"
<end_of_turn>
<start_of_turn>model
"""

    # 5. Run inference with strict JSON schema enforcement
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        response_format={
            "type": "json_object",
            "schema": ClassificationResponse.model_json_schema()
        },
        temperature=0.0  # Keep it deterministic for pipelines
    )

    # 6. Parse the guaranteed JSON output
    raw_content = response["choices"][0]["message"]["content"]
    parsed_json = json.loads(raw_content)

    return parsed_json["category"], parsed_json["confidence"]

In [5]:
# 1. Define your dynamic codebook / taxonomy
my_categories = {
    "Billing / Invoicing": "Questions about credit cards, unexpected charges, refunds, or payment terms.",
    "Technical Support": "Bug reports, application crashes, login issues, or API connectivity errors.",
    "Feature Request": "Users asking for new tools, enhancements, integrations, or UI changes.",
    "Spam / Irrelevant": "Gibberish, marketing pitches, or text completely unrelated to the platform."
}

# 2. Sample text from a pipeline dataset
sample_text = "I am getting a 403 Forbidden error every time I try to ping the webhooks endpoint."

# 3. Execute classification
category, confidence = classify_text(llm, my_categories, sample_text)

print(f"Assigned Category: {category}")
print(f"Confidence Score:  {confidence}")

Assigned Category: Technical Support
Confidence Score:  1.0
